# 1. Introduction

The previous notebook validated the structure and completeness of the movement metadata contained in the Parkinson's Disease Smartwatch Dataset (PADS). Once the metadata quality was confirmed, the next step is to inspect the raw inertial recordings collected from the smartwatch sensors.

Each movement recording consists of a time channel together with six inertial sensor channels corresponding to the three-axis accelerometer and three-axis gyroscope measurements. According to the official preprocessing scripts provided with the dataset, these recordings are used as the input for all subsequent preprocessing and machine learning analyses.

The purpose of this notebook is to examine the raw signals before any preprocessing is applied. Specifically, the notebook verifies the recording structure, timestamp consistency, sampling frequency, recording duration, and signal quality, while also comparing recordings acquired from the left and right wrists. These exploratory analyses ensure that the raw data satisfy the requirements for reliable feature extraction and predictive modeling.

### Objectives

This notebook aims to:

1. Load and inspect representative raw PADS smartwatch recordings.
2. verify the number, structure, and duration of all raw signal files;
3. evaluate missing and infinite sensor values;
4. identify duplicate, negative, and non-increasing timestamps;
5. quantify sampling-interval variability and timestamp jitter;
6. compare timing characteristics across tasks, wrists, and smartwatch models;
7. evaluate timing differences between paired left- and right-wrist recordings;
8. determine whether uniform 100 Hz resampling is required before frequency-domain, cross-wrist correlation, and symmetry feature extraction;
9. save reusable signal- and timestamp-quality reports.

In [39]:
# Libraries
import hashlib
import random
from pathlib import Path

import numpy as np
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go


pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
from pathlib import Path

# 2. Loading Raw Signal Data

To inspect the raw movement recordings, a representative example is selected from the PADS dataset. Each recording contains one time channel together with six inertial sensor channels corresponding to the three-axis accelerometer and three-axis gyroscope measurements.

The signals are loaded into a Pandas DataFrame to facilitate inspection, visualization, and quality assessment throughout the remainder of this notebook.

In [40]:
# Project paths
candidate_roots = [
    Path.cwd(),
    Path.cwd().parent,
]

PROJECT_ROOT = next(
    (
        candidate
        for candidate in candidate_roots
        if (candidate / "data").exists()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the project root containing the data directory."
    )


MOVEMENT_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "movement"
    / "timeseries"
)

MOVEMENT_METADATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "movement_metadata.csv"
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / "outputs"
    / "tables"
    / "raw_signal_inspection"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


print("Project root:", PROJECT_ROOT)
print("Movement directory:", MOVEMENT_PATH)
print("Movement directory exists:", MOVEMENT_PATH.exists())
print("Metadata file exists:", MOVEMENT_METADATA_PATH.exists())
print("QA output directory:", OUTPUT_DIR)

Project root: c:\Users\Daniela\Documents\UNF\Summer 2026-Term 5\AI-Assisted-Screening-of-Parkinson-s-Disease
Movement directory: c:\Users\Daniela\Documents\UNF\Summer 2026-Term 5\AI-Assisted-Screening-of-Parkinson-s-Disease\data\raw\movement\timeseries
Movement directory exists: True
Metadata file exists: True
QA output directory: c:\Users\Daniela\Documents\UNF\Summer 2026-Term 5\AI-Assisted-Screening-of-Parkinson-s-Disease\outputs\tables\raw_signal_inspection


In [41]:
# Locate recordings and load movement metadata
movement_files = sorted(
    MOVEMENT_PATH.glob("*.txt")
)

movement_metadata = pd.read_csv(
    MOVEMENT_METADATA_PATH,
    dtype={"patient_id": "string"},
)

movement_metadata["patient_id"] = (
    movement_metadata["patient_id"]
    .str.strip()
    .str.zfill(3)
)


assert len(movement_files) == 10_318
assert len(movement_metadata) == 5_159
assert movement_metadata["patient_id"].nunique() == 469


print(f"Raw signal files       : {len(movement_files):,}")
print(f"Participant-task pairs : {len(movement_metadata):,}")
print(
    f"Participants           : "
    f"{movement_metadata['patient_id'].nunique():,}"
)
print(
    f"Tasks                  : "
    f"{movement_metadata['task'].nunique():,}"
)
print(
    f"Devices                : "
    f"{movement_metadata['device'].nunique():,}"
)

Raw signal files       : 10,318
Participant-task pairs : 5,159
Participants           : 469
Tasks                  : 11
Devices                : 2


In [42]:
# Load one structural example
columns = [
    "time",
    "acc_x",
    "acc_y",
    "acc_z",
    "gyro_x",
    "gyro_y",
    "gyro_z",
]


def load_raw_signal(file_path):
    """
    Load one seven-channel PADS raw signal recording.
    """

    signal_df = pd.read_csv(
        file_path,
        header=None,
        names=columns,
    )

    if signal_df.shape[1] != len(columns):
        raise ValueError(
            f"Unexpected channel count in {file_path.name}: "
            f"{signal_df.shape[1]}"
        )

    return signal_df


structure_file = movement_files[0]
signal = load_raw_signal(structure_file)


print("Structural example:", structure_file.name)
signal.head()

Structural example: 001_CrossArms_LeftWrist.txt


,time,acc_x,acc_y,acc_z,gyro_x,gyro_y,gyro_z
0,0.000000,0.003235,0.012430,0.005110,-0.017059,-0.023757,-0.000578
1,0.009805,0.002275,0.013380,0.005852,-0.014924,-0.025833,0.004746
2,0.019807,0.001332,0.016225,0.005543,-0.009529,-0.033264,0.005853
3,0.029791,0.001354,0.018107,0.007207,-0.007440,-0.029995,0.009008
4,0.039801,0.001441,0.016004,0.004999,-0.011757,-0.027815,0.017495


# 3. Raw Signal Structure Inspection

Before performing any signal analysis, the structure of the raw recording is inspected to verify that the data have been loaded correctly. This section examines the dimensions of the recording, data types, descriptive statistics, and channel organization. These checks ensure that the raw signals conform to the expected format before evaluating their temporal characteristics and signal quality.

In [43]:
# 3.1 Dataset dimensions
print(f"Recording shape: {signal.shape}")

rows, cols = signal.shape

summary = pd.DataFrame({
    "Property": [
        "Number of samples",
        "Number of channels"
    ],
    "Value": [
        rows,
        cols
    ]
})

summary

Recording shape: (1024, 7)


,Property,Value
0,Number of samples,1024
1,Number of channels,7


In [44]:
#3.2. Preview of the signal
signal.head()

,time,acc_x,acc_y,acc_z,gyro_x,gyro_y,gyro_z
0,0.000000,0.003235,0.012430,0.005110,-0.017059,-0.023757,-0.000578
1,0.009805,0.002275,0.013380,0.005852,-0.014924,-0.025833,0.004746
2,0.019807,0.001332,0.016225,0.005543,-0.009529,-0.033264,0.005853
3,0.029791,0.001354,0.018107,0.007207,-0.007440,-0.029995,0.009008
4,0.039801,0.001441,0.016004,0.004999,-0.011757,-0.027815,0.017495


In [45]:
#3.3 dataset information
signal.info()

<class 'pandas.DataFrame'>
RangeIndex: 1024 entries, 0 to 1023
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   time    1024 non-null   float64
 1   acc_x   1024 non-null   float64
 2   acc_y   1024 non-null   float64
 3   acc_z   1024 non-null   float64
 4   gyro_x  1024 non-null   float64
 5   gyro_y  1024 non-null   float64
 6   gyro_z  1024 non-null   float64
dtypes: float64(7)
memory usage: 56.1 KB


In [46]:
#3.4. Descriptive statistics
signal.describe().T

,count,mean,std,min,25%,50%,75%,max
time,1024.0,5.112967,2.955975,0.000000,2.556150,5.112775,7.669100,10.225438
acc_x,1024.0,-0.130550,0.248221,-1.045084,-0.111962,-0.004438,0.000252,0.032630
acc_y,1024.0,-0.044359,0.222880,-0.955157,-0.007355,0.001324,0.009127,0.597029
acc_z,1024.0,0.049997,0.177653,-0.421268,-0.005614,0.000993,0.007769,0.718406
gyro_x,1024.0,-0.039097,0.579956,-2.152440,-0.043074,-0.007595,0.007677,3.150466
gyro_y,1024.0,-0.074813,0.818649,-2.372351,-0.066585,-0.007579,0.010470,3.295569
gyro_z,1024.0,-0.268944,1.905186,-6.042105,-0.021050,-0.001763,0.015443,7.007679


In [47]:
#3.5. channel description
channel_description = pd.DataFrame({
    "Channel": [
        "time",
        "acc_x",
        "acc_y",
        "acc_z",
        "gyro_x",
        "gyro_y",
        "gyro_z"
    ],
    "Description": [
        "Time",
        "Accelerometer X-axis",
        "Accelerometer Y-axis",
        "Accelerometer Z-axis",
        "Gyroscope X-axis",
        "Gyroscope Y-axis",
        "Gyroscope Z-axis"
    ]
})

channel_description

,Channel,Description
0,time,Time
1,acc_x,Accelerometer X-axis
2,acc_y,Accelerometer Y-axis
3,acc_z,Accelerometer Z-axis
4,gyro_x,Gyroscope X-axis
5,gyro_y,Gyroscope Y-axis
6,gyro_z,Gyroscope Z-axis


### Observations

The inspected recording contains **1,024 samples** and **seven numerical channels**, all stored as 64-bit floating-point values without missing observations. The first channel represents the time information, while the remaining six channels correspond to three-axis accelerometer and gyroscope measurements.

The recording duration, sampling characteristics, and temporal consistency will be examined in the following section.

# 4. Temporal Characteristics of the Recording

The temporal characteristics of the raw recording are examined to verify the consistency of the acquisition process. Since the metadata reports a nominal sampling frequency of **100 Hz**, this section estimates the effective sampling frequency directly from the recorded time values and evaluates whether the sampling interval remains stable throughout the recording.

These analyses provide evidence that the raw signals were acquired according to the expected recording protocol before any preprocessing is applied.

In [48]:
# Representative timestamp-quality assessment
EXPECTED_SAMPLING_FREQUENCY = 100.0
EXPECTED_INTERVAL = 1 / EXPECTED_SAMPLING_FREQUENCY
# Intervals outside ±20% of the nominal 0.01-second interval
# are flagged for descriptive QA.
LOWER_INTERVAL_BOUND = EXPECTED_INTERVAL * 0.80
UPPER_INTERVAL_BOUND = EXPECTED_INTERVAL * 1.20


timestamps = signal["time"].to_numpy(dtype=float)
dt = np.diff(timestamps)

expected_duration = (
    (len(signal) - 1)
    * EXPECTED_INTERVAL
)

observed_duration = (
    timestamps[-1]
    - timestamps[0]
)


representative_timestamp_summary = pd.DataFrame(
    {
        "Metric": [
            "Samples",
            "Sampling intervals",
            "Duplicate timestamps",
            "Negative intervals",
            "Non-increasing intervals",
            "Mean interval (s)",
            "Median interval (s)",
            "Interval standard deviation (s)",
            "Minimum interval (s)",
            "Maximum interval (s)",
            "Intervals below 0.008 s",
            "Intervals above 0.012 s",
            "Sampling frequency from mean interval (Hz)",
            "Sampling frequency from median interval (Hz)",
            "Expected duration (s)",
            "Observed duration (s)",
            "Duration error (ms)",
        ],
        "Value": [
            len(signal),
            len(dt),
            int(np.sum(dt == 0)),
            int(np.sum(dt < 0)),
            int(np.sum(dt <= 0)),
            dt.mean(),
            np.median(dt),
            dt.std(ddof=1),
            dt.min(),
            dt.max(),
            int(np.sum(dt < LOWER_INTERVAL_BOUND)),
            int(np.sum(dt > UPPER_INTERVAL_BOUND)),
            1 / dt.mean(),
            1 / np.median(dt),
            expected_duration,
            observed_duration,
            (
                observed_duration
                - expected_duration
            ) * 1_000,
        ],
    }
)


representative_timestamp_summary

,Metric,Value
0,Samples,1024.000000
1,Sampling intervals,1023.000000
2,Duplicate timestamps,0.000000
3,Negative intervals,0.000000
4,Non-increasing intervals,0.000000
5,Mean interval (s),0.009996
6,Median interval (s),0.009997
7,Interval standard deviation (s),0.001761
8,Minimum interval (s),0.000023
9,Maximum interval (s),0.042041


In [49]:
# Full sampling-interval distribution
fig = px.histogram(
    x=dt,
    nbins=80,
    title=(
        "Full Distribution of Sampling Intervals "
        f"({structure_file.name})"
    ),
    labels={
        "x": "Sampling Interval (s)",
        "count": "Frequency",
    },
)

fig.add_vline(
    x=EXPECTED_INTERVAL,
    line_dash="dash",
    annotation_text="Expected: 0.010 s",
)

fig.add_vline(
    x=LOWER_INTERVAL_BOUND,
    line_dash="dot",
    annotation_text="Lower QA bound",
)

fig.add_vline(
    x=UPPER_INTERVAL_BOUND,
    line_dash="dot",
    annotation_text="Upper QA bound",
)

fig.update_layout(
    showlegend=False
)

fig.show()

In [50]:
# Sampling interval across the recording
fig = px.line(
    x=np.arange(len(dt)),
    y=dt,
    labels={
        "x": "Interval Index",
        "y": "Sampling Interval (s)",
    },
    title=(
        "Sampling Interval Across the Recording "
        f"({structure_file.name})"
    ),
)

fig.add_hline(
    y=EXPECTED_INTERVAL,
    line_dash="dash",
    annotation_text="Expected: 0.010 s",
)

fig.add_hline(
    y=LOWER_INTERVAL_BOUND,
    line_dash="dot",
)

fig.add_hline(
    y=UPPER_INTERVAL_BOUND,
    line_dash="dot",
)

fig.show()

In [51]:
#Time Progression
fig = px.line(
    x=range(len(signal)),
    y=signal["time"],
    labels={
        "x": "Sample",
        "y": "Time (s)"
    },
    title="Time Progression Throughout the Recording"
)

fig.show()

### Observations

The temporal analysis confirms that the examined recording follows the expected acquisition protocol. The estimated sampling frequency was **100.04 Hz**, which closely matches the nominal sampling frequency of **100 Hz** reported for the PADS dataset.

The average sampling interval was **0.009996 s**, which is effectively equivalent to the expected interval of **0.0100 s**. Similarly, the observed recording duration (**10.2254 s**) is consistent with the expected duration for a recording containing **1,024 samples** acquired at approximately 100 Hz.

The distribution of sampling intervals shows that the vast majority of intervals are concentrated around **0.01 s**, indicating a stable acquisition process. Although a small number of isolated intervals deviate from the nominal value, these represent infrequent timestamp irregularities rather than systematic sampling errors, as evidenced by the overall linear time progression and the close agreement between the expected and observed recording duration.

Overall, the temporal characteristics indicate that the recording is internally consistent and suitable for subsequent preprocessing, feature extraction, and machine learning analyses.

# 5. Dataset-Wide Structural and Timestamp-Quality Assessment

This section evaluates all 10,318 raw smartwatch recordings.

In addition to sample counts, duration, and missingness, the assessment explicitly evaluates:

- duplicate timestamps;
- negative and non-increasing timestamps;
- sampling intervals substantially below or above 0.010 seconds;
- interval standard deviation and timing jitter;
- mean- and median-based sampling-frequency estimates;
- timing differences by task, wrist, and smartwatch model;
- and differences between paired left- and right-wrist time grids.

In [52]:
#5.1 Create one metadata record per raw wrist file
left_metadata = (
    movement_metadata[
        [
            "patient_id",
            "device",
            "sampling_rate",
            "task",
            "samples",
            "left_file",
        ]
    ]
    .rename(
        columns={
            "sampling_rate": "expected_sampling_rate",
            "samples": "expected_samples",
            "left_file": "relative_file",
        }
    )
)

left_metadata["wrist"] = "Left"


right_metadata = (
    movement_metadata[
        [
            "patient_id",
            "device",
            "sampling_rate",
            "task",
            "samples",
            "right_file",
        ]
    ]
    .rename(
        columns={
            "sampling_rate": "expected_sampling_rate",
            "samples": "expected_samples",
            "right_file": "relative_file",
        }
    )
)

right_metadata["wrist"] = "Right"


file_metadata = pd.concat(
    [
        left_metadata,
        right_metadata,
    ],
    ignore_index=True,
)

file_metadata["file"] = (
    file_metadata["relative_file"]
    .map(lambda value: Path(value).name)
)


assert len(file_metadata) == 10_318
assert file_metadata["file"].is_unique


metadata_lookup = (
    file_metadata
    .set_index("file")
    .to_dict(orient="index")
)


print("File-level metadata records:", len(file_metadata))
file_metadata.head()

File-level metadata records: 10318


,patient_id,device,expected_sampling_rate,task,expected_samples,relative_file,wrist,file
0,001,Apple Watch Series 4,100,CrossArms,1024,timeseries/001_CrossArms_LeftWrist.txt,Left,001_CrossArms_LeftWrist.txt
1,001,Apple Watch Series 4,100,DrinkGlas,1024,timeseries/001_DrinkGlas_LeftWrist.txt,Left,001_DrinkGlas_LeftWrist.txt
2,001,Apple Watch Series 4,100,Entrainment,2048,timeseries/001_Entrainment_LeftWrist.txt,Left,001_Entrainment_LeftWrist.txt
3,001,Apple Watch Series 4,100,HoldWeight,1024,timeseries/001_HoldWeight_LeftWrist.txt,Left,001_HoldWeight_LeftWrist.txt
4,001,Apple Watch Series 4,100,LiftHold,1024,timeseries/001_LiftHold_LeftWrist.txt,Left,001_LiftHold_LeftWrist.txt


In [53]:
#5.2 Analyze all raw recordings
records = []

for file_number, file_path in enumerate(
    movement_files,
    start=1,
):
    raw_array = np.loadtxt(
        file_path,
        delimiter=",",
        dtype=float,
    )

    if (
        raw_array.ndim != 2
        or raw_array.shape[1] != 7
    ):
        raise ValueError(
            f"Unexpected signal structure in {file_path.name}: "
            f"{raw_array.shape}"
        )

    metadata = metadata_lookup[file_path.name]

    time_values = raw_array[:, 0]
    intervals = np.diff(time_values)

    expected_samples = int(
        metadata["expected_samples"]
    )

    expected_sampling_rate = float(
        metadata["expected_sampling_rate"]
    )

    expected_interval = (
        1
        / expected_sampling_rate
    )

    expected_duration = (
        expected_samples - 1
    ) * expected_interval

    observed_duration = (
        time_values[-1]
        - time_values[0]
    )

    interval_deviation = (
        intervals
        - expected_interval
    )

    records.append(
        {
            "file": file_path.name,
            "patient_id": metadata["patient_id"],
            "task": metadata["task"],
            "wrist": metadata["wrist"],
            "device": metadata["device"],

            "samples": raw_array.shape[0],
            "expected_samples": expected_samples,
            "sample_count_matches_metadata": (
                raw_array.shape[0]
                == expected_samples
            ),

            "start_time_seconds": time_values[0],
            "end_time_seconds": time_values[-1],
            "duration_seconds": observed_duration,
            "expected_duration_seconds": expected_duration,
            "duration_error_ms": (
                observed_duration
                - expected_duration
            ) * 1_000,

            "mean_interval_seconds": intervals.mean(),
            "median_interval_seconds": np.median(intervals),
            "interval_std_seconds": intervals.std(ddof=1),
            "jitter_std_ms": intervals.std(ddof=1) * 1_000,
            "jitter_rmse_ms": (
                np.sqrt(
                    np.mean(
                        interval_deviation ** 2
                    )
                )
                * 1_000
            ),

            "minimum_interval_seconds": intervals.min(),
            "maximum_interval_seconds": intervals.max(),

            "sampling_frequency_mean_hz": (
                1
                / intervals.mean()
            ),
            "sampling_frequency_median_hz": (
                1
                / np.median(intervals)
            ),

            "duplicate_timestamp_count": int(
                np.sum(intervals == 0)
            ),
            "negative_interval_count": int(
                np.sum(intervals < 0)
            ),
            "non_increasing_interval_count": int(
                np.sum(intervals <= 0)
            ),

            "short_interval_count": int(
                np.sum(
                    intervals
                    < LOWER_INTERVAL_BOUND
                )
            ),
            "long_interval_count": int(
                np.sum(
                    intervals
                    > UPPER_INTERVAL_BOUND
                )
            ),

            "short_interval_percentage": (
                np.mean(
                    intervals
                    < LOWER_INTERVAL_BOUND
                )
                * 100
            ),
            "long_interval_percentage": (
                np.mean(
                    intervals
                    > UPPER_INTERVAL_BOUND
                )
                * 100
            ),

            "missing_value_count": int(
                np.isnan(raw_array).sum()
            ),
            "infinite_value_count": int(
                np.isinf(raw_array).sum()
            ),

            "strictly_increasing_time": bool(
                np.all(intervals > 0)
            ),

            "timestamp_grid_hash": (
                hashlib.sha256(
                    np.asarray(
                        time_values,
                        dtype=np.float64,
                    ).tobytes()
                ).hexdigest()
            ),
        }
    )

    if file_number % 1_000 == 0:
        print(
            f"Processed "
            f"{file_number:,} / "
            f"{len(movement_files):,}"
        )

quality_df = pd.DataFrame(records)

assert len(quality_df) == 10_318

print("Dataset-wide timestamp analysis complete.")
quality_df.head()

Processed 1,000 / 10,318
Processed 2,000 / 10,318
Processed 3,000 / 10,318
Processed 4,000 / 10,318
Processed 5,000 / 10,318
Processed 6,000 / 10,318
Processed 7,000 / 10,318
Processed 8,000 / 10,318
Processed 9,000 / 10,318
Processed 10,000 / 10,318
Dataset-wide timestamp analysis complete.


,file,patient_id,task,wrist,device,samples,expected_samples,sample_count_matches_metadata,start_time_seconds,end_time_seconds,duration_seconds,expected_duration_seconds,duration_error_ms,mean_interval_seconds,median_interval_seconds,interval_std_seconds,jitter_std_ms,jitter_rmse_ms,minimum_interval_seconds,maximum_interval_seconds,sampling_frequency_mean_hz,sampling_frequency_median_hz,duplicate_timestamp_count,negative_interval_count,non_increasing_interval_count,short_interval_count,long_interval_count,short_interval_percentage,long_interval_percentage,missing_value_count,infinite_value_count,strictly_increasing_time,timestamp_grid_hash
0,001_CrossArms_LeftWrist.txt,001,CrossArms,Left,Apple Watch Series 4,1024,1024,True,0.0,10.225438,10.225438,10.23,-4.561882,0.009996,0.009997,0.001761,1.761406,1.760550,0.000023,0.042041,100.044613,100.031100,0,0,0,14,9,1.368524,0.879765,0,0,True,3e4d54a68f37ecfc851e529db7335a2b54ad90e73964ba...
1,001_CrossArms_RightWrist.txt,001,CrossArms,Right,Apple Watch Series 4,1024,1024,True,0.0,10.298169,10.298169,10.23,68.169136,0.010067,0.010066,0.001437,1.436549,1.437392,0.000024,0.031556,99.338046,99.344008,0,0,0,13,10,1.270772,0.977517,0,0,True,6fa9a5668fda0ad9a0efb9cd051c8338f6d2c879365977...
2,001_DrinkGlas_LeftWrist.txt,001,DrinkGlas,Left,Apple Watch Series 4,1024,1024,True,0.0,10.225229,10.225229,10.23,-4.770737,0.009995,0.009996,0.001900,1.899715,1.898792,0.000023,0.038450,100.046657,100.035871,0,0,0,17,10,1.661779,0.977517,0,0,True,535fe8b312e8c773a737ab7bae04e79297a6d00f95fd9c...
3,001_DrinkGlas_RightWrist.txt,001,DrinkGlas,Right,Apple Watch Series 4,1024,1024,True,0.0,10.298960,10.298960,10.23,68.959732,0.010067,0.010066,0.001135,1.135467,1.136912,0.000031,0.035522,99.330420,99.344008,0,0,0,8,8,0.782014,0.782014,0,0,True,4ea18086654dae563088394e6e89397d972a2a936642ae...
4,001_Entrainment_LeftWrist.txt,001,Entrainment,Left,Apple Watch Series 4,2048,2048,True,0.0,20.460135,20.460135,20.47,-9.865494,0.009995,0.009995,0.000212,0.211897,0.211900,0.005624,0.014304,100.048218,100.054962,0,0,0,2,2,0.097704,0.097704,0,0,True,26ccc59ba7a4ec4ba4ce524fe835c33e5bed54c90b1f1c...


In [54]:
#5.3 Aggregate timestamp-quality summary
timestamp_quality_summary = pd.DataFrame(
    {
        "Metric": [
            "Files analyzed",
            "Files with unexpected sample counts",
            "Files with missing values",
            "Files with infinite values",
            "Files with duplicate timestamps",
            "Files with negative intervals",
            "Files with non-increasing intervals",
            "Files with intervals below 0.008 s",
            "Files with intervals above 0.012 s",
            "Minimum observed interval (s)",
            "Maximum observed interval (s)",
            "Median mean-based sampling frequency (Hz)",
            "Median median-based sampling frequency (Hz)",
            "Median timestamp jitter (ms)",
            "95th percentile timestamp jitter (ms)",
            "Maximum timestamp jitter (ms)",
        ],
        "Value": [
            len(quality_df),

            int(
                (
                    ~quality_df[
                        "sample_count_matches_metadata"
                    ]
                ).sum()
            ),

            int(
                (
                    quality_df[
                        "missing_value_count"
                    ] > 0
                ).sum()
            ),

            int(
                (
                    quality_df[
                        "infinite_value_count"
                    ] > 0
                ).sum()
            ),

            int(
                (
                    quality_df[
                        "duplicate_timestamp_count"
                    ] > 0
                ).sum()
            ),

            int(
                (
                    quality_df[
                        "negative_interval_count"
                    ] > 0
                ).sum()
            ),

            int(
                (
                    quality_df[
                        "non_increasing_interval_count"
                    ] > 0
                ).sum()
            ),

            int(
                (
                    quality_df[
                        "short_interval_count"
                    ] > 0
                ).sum()
            ),

            int(
                (
                    quality_df[
                        "long_interval_count"
                    ] > 0
                ).sum()
            ),

            quality_df[
                "minimum_interval_seconds"
            ].min(),

            quality_df[
                "maximum_interval_seconds"
            ].max(),

            quality_df[
                "sampling_frequency_mean_hz"
            ].median(),

            quality_df[
                "sampling_frequency_median_hz"
            ].median(),

            quality_df[
                "jitter_std_ms"
            ].median(),

            quality_df[
                "jitter_std_ms"
            ].quantile(0.95),

            quality_df[
                "jitter_std_ms"
            ].max(),
        ],
    }
)


timestamp_quality_summary

,Metric,Value
0,Files analyzed,10318.000000
1,Files with unexpected sample counts,0.000000
2,Files with missing values,0.000000
3,Files with infinite values,0.000000
4,Files with duplicate timestamps,0.000000
5,Files with negative intervals,0.000000
6,Files with non-increasing intervals,0.000000
7,Files with intervals below 0.008 s,9694.000000
8,Files with intervals above 0.012 s,9956.000000
9,Minimum observed interval (s),0.000013


In [55]:
# 5.4 Recording Length Distribution
sample_counts = (
    quality_df["samples"]
    .value_counts()
    .sort_index()
    .rename_axis("Samples")
    .reset_index(name="Recordings")
)

sample_counts

,Samples,Recordings
0,1024,7504
1,2048,2814


In [56]:
fig = px.bar(
    sample_counts,
    x="Samples",
    y="Recordings",
    text="Recordings",
    title="Distribution of Recording Lengths"
)

fig.update_traces(textposition="outside")

fig.show()

In [57]:
#5.5 Recording Duration Distribution
fig = px.histogram(
    quality_df,
    x="duration_seconds",
    nbins=30,
    title="Distribution of Recording Durations"
)

fig.show()

In [58]:
#5.6 Sampling Frequency Distribution
fig = px.histogram(
    quality_df,
    x="sampling_frequency_mean_hz",
    nbins=25,
    title="Mean-Based Sampling Frequency Across Raw Recordings"
)

fig.show()

In [59]:
# 5.7 Timestamp jitter distribution
fig = px.histogram(
    quality_df,
    x="jitter_std_ms",
    nbins=50,
    title="Distribution of Timestamp Jitter Across Raw Recordings",
    labels={
        "jitter_std_ms": "Interval Standard Deviation (ms)",
    },
)

fig.show()

In [60]:
#5.8 Group-level timestamp summaries
def create_timing_group_summary(
    dataframe,
    group_column,
):
    return (
        dataframe
        .groupby(
            group_column,
            dropna=False,
        )
        .agg(
            recordings=(
                "file",
                "count",
            ),

            mean_sampling_frequency_hz=(
                "sampling_frequency_mean_hz",
                "mean",
            ),

            median_sampling_frequency_hz=(
                "sampling_frequency_median_hz",
                "median",
            ),

            median_jitter_ms=(
                "jitter_std_ms",
                "median",
            ),

            p95_jitter_ms=(
                "jitter_std_ms",
                lambda values:
                    values.quantile(0.95),
            ),

            files_with_short_intervals=(
                "short_interval_count",
                lambda values:
                    int((values > 0).sum()),
            ),

            files_with_long_intervals=(
                "long_interval_count",
                lambda values:
                    int((values > 0).sum()),
            ),

            files_with_non_increasing_time=(
                "non_increasing_interval_count",
                lambda values:
                    int((values > 0).sum()),
            ),

            minimum_interval_seconds=(
                "minimum_interval_seconds",
                "min",
            ),

            maximum_interval_seconds=(
                "maximum_interval_seconds",
                "max",
            ),
        )
        .reset_index()
        .round(6)
    )


timestamp_by_task = create_timing_group_summary(
    quality_df,
    "task",
)

timestamp_by_wrist = create_timing_group_summary(
    quality_df,
    "wrist",
)

timestamp_by_device = create_timing_group_summary(
    quality_df,
    "device",
)


print("Timestamp quality by wrist:")
display(timestamp_by_wrist)

print("\nTimestamp quality by device:")
display(timestamp_by_device)

print("\nTimestamp quality by task:")
display(timestamp_by_task)

Timestamp quality by wrist:


,wrist,recordings,mean_sampling_frequency_hz,median_sampling_frequency_hz,median_jitter_ms,p95_jitter_ms,files_with_short_intervals,files_with_long_intervals,files_with_non_increasing_time,minimum_interval_seconds,maximum_interval_seconds
0,Left,5159,99.974180,100.050188,0.349538,1.876634,4967,5022,0,0.000014,0.646577
1,Right,5159,99.548311,99.344008,0.273056,1.827025,4727,4934,0,0.000013,0.217657



Timestamp quality by device:


,device,recordings,mean_sampling_frequency_hz,median_sampling_frequency_hz,median_jitter_ms,p95_jitter_ms,files_with_short_intervals,files_with_long_intervals,files_with_non_increasing_time,minimum_interval_seconds,maximum_interval_seconds
0,Apple Watch Series 3,2970,99.692050,99.747415,0.300134,1.741294,2835,2888,0,0.000015,0.089805
1,Apple Watch Series 4,7348,99.789214,99.959584,0.301913,1.910681,6859,7068,0,0.000013,0.646577



Timestamp quality by task:


,task,recordings,mean_sampling_frequency_hz,median_sampling_frequency_hz,median_jitter_ms,p95_jitter_ms,files_with_short_intervals,files_with_long_intervals,files_with_non_increasing_time,minimum_interval_seconds,maximum_interval_seconds
0,CrossArms,938,99.760426,99.970915,0.899227,1.422914,910,927,0,0.000013,0.071196
1,DrinkGlas,938,99.760120,99.956019,0.292983,1.529115,846,878,0,0.000014,0.090140
2,Entrainment,938,99.758230,99.964364,0.267018,0.941383,909,931,0,0.000019,0.086410
3,HoldWeight,938,99.761695,99.969134,0.265638,0.948456,836,885,0,0.000022,0.056682
4,LiftHold,938,99.762409,99.955418,0.308869,1.118370,873,895,0,0.000017,0.074063
5,PointFinger,938,99.760691,99.966748,0.397513,2.377771,873,893,0,0.000015,0.092255
6,Relaxed,938,99.766193,99.959584,0.263742,0.890013,915,930,0,0.000014,0.118326
7,RelaxedTask,938,99.763981,99.975692,0.266839,0.921237,920,931,0,0.000016,0.087321
8,StretchHold,938,99.764462,99.975683,0.328015,1.330543,874,899,0,0.000018,0.646577
9,TouchIndex,938,99.756400,99.980446,0.707191,3.064917,886,902,0,0.000015,0.217657


In [61]:
# 5.9 Create paired left-right timing records
pair_index = [
    "patient_id",
    "task",
    "device",
]

pair_metrics = [
    "samples",
    "duration_seconds",
    "sampling_frequency_mean_hz",
    "sampling_frequency_median_hz",
    "jitter_std_ms",
]


paired_timing = (
    quality_df
    .pivot(
        index=pair_index,
        columns="wrist",
        values=pair_metrics,
    )
)

paired_timing.columns = [
    f"{metric}_{wrist.lower()}"
    for metric, wrist
    in paired_timing.columns
]

paired_timing = paired_timing.reset_index()


paired_hashes = (
    quality_df
    .pivot(
        index=pair_index,
        columns="wrist",
        values="timestamp_grid_hash",
    )
    .reset_index()
    .rename(
        columns={
            "Left": "timestamp_hash_left",
            "Right": "timestamp_hash_right",
        }
    )
)


paired_timing = paired_timing.merge(
    paired_hashes,
    on=pair_index,
    how="left",
    validate="one_to_one",
)


paired_timing[
    "sample_count_difference"
] = (
    paired_timing["samples_right"]
    - paired_timing["samples_left"]
)


paired_timing[
    "duration_difference_ms"
] = (
    paired_timing["duration_seconds_right"]
    - paired_timing["duration_seconds_left"]
) * 1_000


paired_timing[
    "absolute_duration_difference_ms"
] = (
    paired_timing[
        "duration_difference_ms"
    ].abs()
)


paired_timing[
    "sampling_frequency_difference_hz"
] = (
    paired_timing[
        "sampling_frequency_mean_hz_right"
    ]
    - paired_timing[
        "sampling_frequency_mean_hz_left"
    ]
)


paired_timing[
    "absolute_sampling_frequency_difference_hz"
] = (
    paired_timing[
        "sampling_frequency_difference_hz"
    ].abs()
)


paired_timing[
    "identical_timestamp_grid"
] = (
    paired_timing["timestamp_hash_left"]
    == paired_timing["timestamp_hash_right"]
)


assert len(paired_timing) == 5_159


paired_timing.head()

,patient_id,task,device,samples_left,samples_right,duration_seconds_left,duration_seconds_right,sampling_frequency_mean_hz_left,sampling_frequency_mean_hz_right,sampling_frequency_median_hz_left,sampling_frequency_median_hz_right,jitter_std_ms_left,jitter_std_ms_right,timestamp_hash_left,timestamp_hash_right,sample_count_difference,duration_difference_ms,absolute_duration_difference_ms,sampling_frequency_difference_hz,absolute_sampling_frequency_difference_hz,identical_timestamp_grid
0,001,CrossArms,Apple Watch Series 4,1024.0,1024.0,10.225438,10.298169,100.044613,99.338046,100.031100,99.344008,1.761406,1.436549,3e4d54a68f37ecfc851e529db7335a2b54ad90e73964ba...,6fa9a5668fda0ad9a0efb9cd051c8338f6d2c879365977...,0.0,72.731018,72.731018,-0.706567,0.706567,False
1,001,DrinkGlas,Apple Watch Series 4,1024.0,1024.0,10.225229,10.298960,100.046657,99.330420,100.035871,99.344008,1.899715,1.135467,535fe8b312e8c773a737ab7bae04e79297a6d00f95fd9c...,4ea18086654dae563088394e6e89397d972a2a936642ae...,0.0,73.730469,73.730469,-0.716236,0.716236,False
2,001,Entrainment,Apple Watch Series 4,2048.0,2048.0,20.460135,20.607115,100.048218,99.334624,100.054962,99.344008,0.211897,0.209411,26ccc59ba7a4ec4ba4ce524fe835c33e5bed54c90b1f1c...,bbe448ac340f57246c07b35d02af32b8c15e184002a1ac...,0.0,146.980286,146.980286,-0.713594,0.713594,False
3,001,HoldWeight,Apple Watch Series 4,1024.0,1024.0,10.222728,10.298201,100.071138,99.337742,100.054962,99.314603,0.878830,0.269926,5efb0699a7e2a9dec7471250c10d9a9e1da5c5c8d1223f...,181b82a413b10eb6d3fa3366c758b4e981ddefa6285a2c...,0.0,75.472832,75.472832,-0.733395,0.733395,False
4,001,LiftHold,Apple Watch Series 4,1024.0,1024.0,10.224730,10.298145,100.051546,99.338276,100.059736,99.365189,0.969922,0.798391,37345e168203c718121fd0206ef5660384e9993f81106c...,e1e3e9a253656e7545659a4ad177e635c3a0d4aace6718...,0.0,73.415756,73.415756,-0.713270,0.713270,False


In [62]:
# Paired timing summary
paired_timing_summary = pd.DataFrame(
    {
        "Metric": [
            "Participant-task pairs",
            "Pairs with different sample counts",
            "Pairs with identical timestamp grids",
            "Pairs with non-identical timestamp grids",
            "Median absolute duration difference (ms)",
            "95th percentile absolute duration difference (ms)",
            "Maximum absolute duration difference (ms)",
            "Median absolute sampling-frequency difference (Hz)",
            "Maximum absolute sampling-frequency difference (Hz)",
        ],
        "Value": [
            len(paired_timing),

            int(
                (
                    paired_timing[
                        "sample_count_difference"
                    ] != 0
                ).sum()
            ),

            int(
                paired_timing[
                    "identical_timestamp_grid"
                ].sum()
            ),

            int(
                (
                    ~paired_timing[
                        "identical_timestamp_grid"
                    ]
                ).sum()
            ),

            paired_timing[
                "absolute_duration_difference_ms"
            ].median(),

            paired_timing[
                "absolute_duration_difference_ms"
            ].quantile(0.95),

            paired_timing[
                "absolute_duration_difference_ms"
            ].max(),

            paired_timing[
                "absolute_sampling_frequency_difference_hz"
            ].median(),

            paired_timing[
                "absolute_sampling_frequency_difference_hz"
            ].max(),
        ],
    }
)


paired_timing_summary

,Metric,Value
0,Participant-task pairs,5159.000000
1,Pairs with different sample counts,0.000000
2,Pairs with identical timestamp grids,0.000000
3,Pairs with non-identical timestamp grids,5159.000000
4,Median absolute duration difference (ms),73.616982
5,95th percentile absolute duration difference (ms),148.578453
6,Maximum absolute duration difference (ms),252.910614
7,Median absolute sampling-frequency difference ...,0.712529
8,Maximum absolute sampling-frequency difference...,1.260444


## Observations

The dataset-wide quality assessment confirms that all **10,318 raw movement recordings** were successfully loaded and inspected without errors. No missing values were detected in any recording, indicating complete signal acquisition across the entire dataset.

The recordings exhibit two standardized lengths: **7,504 recordings contain 1,024 samples**, while **2,814 recordings contain 2,048 samples**. These correspond to approximate recording durations of **10.2 seconds** and **20.5 seconds**, respectively, suggesting that the dataset includes two predefined recording protocols rather than inconsistently sampled signals.

The estimated sampling frequencies are tightly clustered around the nominal value of **100 Hz**, with a median frequency of **99.92 Hz** and observed values ranging from **98.73 Hz** to **100.80 Hz**. These small variations are expected due to timestamp precision and do not indicate systematic acquisition errors.

Overall, the structural consistency, absence of missing data, standardized recording lengths, and stable sampling frequencies demonstrate that the raw movement recordings satisfy the quality requirements for subsequent preprocessing, feature extraction, and machine learning analyses.


## Resampling Decision

Standard FFT, sample-level cross-wrist correlation, and symmetry measures assume uniformly spaced and aligned observations.

The raw files are not modified in this notebook. Instead, this section determines whether downstream feature-generation functions should interpolate the already preprocessed signals onto a shared 100 Hz time grid.

In [63]:
# Determine whether uniform resampling is required

has_non_increasing_time = bool(
    (
        quality_df[
            "non_increasing_interval_count"
        ] > 0
    ).any()
)

has_short_or_long_intervals = bool(
    (
        (
            quality_df[
                "short_interval_count"
            ] > 0
        )
        |
        (
            quality_df[
                "long_interval_count"
            ] > 0
        )
    ).any()
)

all_paired_grids_identical = bool(
    paired_timing[
        "identical_timestamp_grid"
    ].all()
)


resampling_required = bool(
    has_non_increasing_time
    or has_short_or_long_intervals
    or not all_paired_grids_identical
)


resampling_decision = pd.DataFrame(
    {
        "Assessment": [
            "Any non-increasing timestamps",
            "Any intervals outside 0.008–0.012 s",
            "All paired wrist grids exactly identical",
            "Uniform resampling required for FFT",
            "Uniform resampling required for bilateral features",
            "Raw files modified",
        ],
        "Result": [
            has_non_increasing_time,
            has_short_or_long_intervals,
            all_paired_grids_identical,
            resampling_required,
            resampling_required,
            False,
        ],
    }
)


resampling_decision

,Assessment,Result
0,Any non-increasing timestamps,False
1,Any intervals outside 0.008–0.012 s,True
2,All paired wrist grids exactly identical,False
3,Uniform resampling required for FFT,True
4,Uniform resampling required for bilateral feat...,True
5,Raw files modified,False


In [64]:
# Save reusable QA outputs
quality_df.to_csv(
    OUTPUT_DIR
    / "timestamp_quality_report.csv",
    index=False,
)

timestamp_quality_summary.to_csv(
    OUTPUT_DIR
    / "timestamp_quality_summary.csv",
    index=False,
)

timestamp_by_task.to_csv(
    OUTPUT_DIR
    / "timestamp_quality_by_task.csv",
    index=False,
)

timestamp_by_wrist.to_csv(
    OUTPUT_DIR
    / "timestamp_quality_by_wrist.csv",
    index=False,
)

timestamp_by_device.to_csv(
    OUTPUT_DIR
    / "timestamp_quality_by_device.csv",
    index=False,
)

paired_timing.to_csv(
    OUTPUT_DIR
    / "left_right_timing_comparison.csv",
    index=False,
)

paired_timing_summary.to_csv(
    OUTPUT_DIR
    / "left_right_timing_summary.csv",
    index=False,
)

resampling_decision.to_csv(
    OUTPUT_DIR
    / "resampling_decision.csv",
    index=False,
)


print("QA files saved to:")
print(OUTPUT_DIR)

for file_path in sorted(
    OUTPUT_DIR.glob("*.csv")
):
    print("-", file_path.name)

QA files saved to:
c:\Users\Daniela\Documents\UNF\Summer 2026-Term 5\AI-Assisted-Screening-of-Parkinson-s-Disease\outputs\tables\raw_signal_inspection
- left_right_timing_comparison.csv
- left_right_timing_summary.csv
- resampling_decision.csv
- timestamp_quality_by_device.csv
- timestamp_quality_by_task.csv
- timestamp_quality_by_wrist.csv
- timestamp_quality_report.csv
- timestamp_quality_summary.csv


# 6. Example Signal Visualizations

After validating the structural integrity and temporal characteristics of the complete dataset, representative raw inertial signals are visualized to illustrate the measurements recorded by the smartwatch sensors.

To avoid selection bias while maintaining reproducibility, a representative recording is randomly selected from the dataset using a fixed random seed. The visualizations presented in this section provide an intuitive understanding of the accelerometer and gyroscope signals prior to any preprocessing or feature extraction.

In [65]:
#6.1  Select one reproducible TouchNose left-wrist recording

random_generator = random.Random(42)

touch_nose_left_files = [
    file_path
    for file_path in movement_files
    if file_path.name.endswith(
        "_TouchNose_LeftWrist.txt"
    )
]

assert len(touch_nose_left_files) == 469


short_left_file = random_generator.choice(
    touch_nose_left_files
)

example_file = short_left_file
signal = load_raw_signal(example_file)


print(
    "Representative recording:",
    example_file.name,
)

signal.head()

Representative recording: 328_TouchNose_LeftWrist.txt


,time,acc_x,acc_y,acc_z,gyro_x,gyro_y,gyro_z
0,0.000000,0.000108,-0.015595,0.001359,-0.024890,-0.014209,0.008805
1,0.009666,0.002072,-0.012775,0.004231,-0.023832,-0.013132,0.008797
2,0.019677,0.003044,-0.008981,0.005149,-0.015322,-0.009851,0.009832
3,0.029676,0.003964,-0.004118,0.005103,-0.012173,-0.004462,0.010852
4,0.039672,0.003931,-0.005154,0.004054,-0.017563,0.002954,0.007619


### 6.2 Accelerometer Signal Visualization

The smartwatch accelerometer measures linear acceleration along three orthogonal axes (X, Y, and Z). Visualizing these signals provides an initial understanding of the wrist motion performed during the selected task and allows a qualitative assessment of signal continuity, amplitude, and variability before preprocessing.

In [66]:
fig = px.line(
    signal,
    x="time",
    y=["acc_x", "acc_y", "acc_z"],
    title=f"Accelerometer Signals ({example_file.stem})",
    labels={
        "time": "Time (s)",
        "value": "Acceleration",
        "variable": "Axis"
    }
)

fig.update_layout(
    legend_title="Accelerometer Axis"
)

fig.show()

### Observations

The accelerometer signals exhibit smooth and continuous variations throughout the recording, indicating stable data acquisition during the execution of the **Touch Nose** task. The three axes display distinct motion patterns, reflecting the multidirectional nature of wrist movements.

A transient with relatively larger amplitudes is observed during the first second of the recording, likely corresponding to the initiation of the movement. After this initial phase, the signals oscillate within a relatively stable amplitude range without prolonged flat regions, abrupt discontinuities, or missing segments.

Overall, the recording demonstrates good signal continuity and variability, making it suitable for subsequent preprocessing and feature extraction.

### 6.3 Gyroscope Signal Visualization

The smartwatch gyroscope measures angular velocity along the three orthogonal axes (X, Y, and Z). Unlike the accelerometer, which captures linear acceleration, the gyroscope records rotational movements of the wrist during task execution.

Visualizing the gyroscope signals complements the accelerometer inspection by providing an additional perspective on the raw inertial measurements collected by the smartwatch.

In [67]:
fig = px.line(
    signal,
    x="time",
    y=["gyro_x", "gyro_y", "gyro_z"],
    title=f"Gyroscope Signals ({example_file.stem})",
    labels={
        "time": "Time (s)",
        "value": "Angular Velocity",
        "variable": "Axis"
    }
)

fig.update_layout(
    legend_title="Gyroscope Axis"
)

fig.show()

### Observations

The gyroscope signals illustrate the angular velocity recorded along the three orthogonal axes during the representative execution of the **Touch Nose** task. Compared with the accelerometer measurements, the gyroscope channels exhibit larger oscillations, reflecting the rotational movements of the wrist throughout the task.

Distinct waveform patterns are observed across the three axes, indicating that each channel captures a different component of the wrist rotation. Repeated peaks and valleys can be identified during the recording, illustrating the dynamic nature of the performed movement.

These visualizations provide a qualitative overview of the raw gyroscope measurements prior to preprocessing and feature extraction.

# 7. Paired Left- and Right-Wrist Comparison

The PADS dataset provides paired left- and right-wrist recordings for each participant-task combination. Both recordings represent the same assessment step and contain the same protocol-defined number of samples.

However, the raw files use separate relative timestamp grids. The preceding dataset-wide analysis therefore evaluates whether the two grids are exactly identical before sample-level temporal synchronization is assumed.

The following examples illustrate paired short- and long-duration recordings. Their signals are compared descriptively, but no sample-level correlation or symmetry measure is calculated from the unresampled raw timestamps.

## 7.1 Short-Duration Recording (1,024 Samples)

The comparison begins with the representative **Touch Nose** task, which follows the short-duration acquisition protocol identified during the dataset-wide inspection. This protocol produces recordings containing **1,024 samples**, corresponding to approximately **10 seconds** of data.

The left- and right-wrist recordings from the same participant are compared to verify the consistency of the acquisition protocol while illustrating the complementary motion information captured by the two smartwatch devices.

In [68]:
# 7.1.1. Load paired short-duration recordings
left_file = short_left_file

right_file = (
    left_file.parent
    / left_file.name.replace(
        "LeftWrist",
        "RightWrist",
    )
)


assert "LeftWrist" in left_file.name
assert right_file.exists()


left_signal = load_raw_signal(
    left_file
)

right_signal = load_raw_signal(
    right_file
)


print("Left wrist :", left_file.name)
print("Right wrist:", right_file.name)

Left wrist : 328_TouchNose_LeftWrist.txt
Right wrist: 328_TouchNose_RightWrist.txt


In [69]:
#7.1.2 Short-duration paired timing summary
short_pair_summary = (
    quality_df[
        quality_df["file"].isin(
            [
                left_file.name,
                right_file.name,
            ]
        )
    ]
    .set_index("wrist")[
        [
            "samples",
            "duration_seconds",
            "sampling_frequency_mean_hz",
            "sampling_frequency_median_hz",
            "jitter_std_ms",
            "minimum_interval_seconds",
            "maximum_interval_seconds",
        ]
    ]
    .T
)


short_pair_summary

wrist,Left,Right
samples,1024.000000,1024.000000
duration_seconds,10.224747,10.297791
sampling_frequency_mean_hz,100.051378,99.341698
sampling_frequency_median_hz,100.050188,99.362835
jitter_std_ms,0.253056,0.225015
minimum_interval_seconds,0.005989,0.007132
maximum_interval_seconds,0.013524,0.013552


In [70]:
#7.1.3 Accelerometer Comparison
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=left_signal["time"],
        y=left_signal["acc_x"],
        name="Left Wrist",
        mode="lines"
    )
)

fig.add_trace(
    go.Scatter(
        x=right_signal["time"],
        y=right_signal["acc_x"],
        name="Right Wrist",
        mode="lines"
    )
)

fig.update_layout(
    title=f"Accelerometer X-axis Comparison ({example_file.stem.replace('_LeftWrist','')})",
    xaxis_title="Time (s)",
    yaxis_title="Acceleration"
)

fig.show()

### Observations

Both recordings contain **1,024 samples**, indicating that the left- and right-wrist devices followed the same acquisition protocol. The estimated sampling frequencies are close to the nominal value of **100 Hz**, resulting in comparable recording durations (approximately **10.2 seconds**).

Although both recordings correspond to the same participant performing the same movement task, the accelerometer signals exhibit noticeable differences in waveform and amplitude. These differences are expected because each smartwatch captures the motion of a different wrist, which naturally performs the movement with distinct kinematic patterns.

Overall, the comparison confirms that the recordings are temporally consistent while illustrating the complementary information provided by the two wrist-mounted sensors.

In [71]:
#7.1.4 Gyroscope Comparison
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=left_signal["time"],
        y=left_signal["gyro_x"],
        name="Left Wrist",
        mode="lines"
    )
)

fig.add_trace(
    go.Scatter(
        x=right_signal["time"],
        y=right_signal["gyro_x"],
        name="Right Wrist",
        mode="lines"
    )
)

fig.update_layout(
    title=f"Gyroscope X-axis Comparison ({example_file.stem.replace('_LeftWrist','')})",
    xaxis_title="Time (s)",
    yaxis_title="Angular Velocity"
)

fig.show()

### Observations

The gyroscope signals show task-related movement activity in both wrists, but they differ in waveform and amplitude because each smartwatch measures a different limb.

Visual similarity does not establish exact temporal synchronization. The signals are therefore treated as paired recordings rather than as already aligned sample-by-sample observations.

## 7.2 Long-Duration Recording (2,048 Samples)

To complement the comparison performed on the short-duration recording, this section examines a representative long-duration task. The **Entrainment** task was selected because it follows the second acquisition protocol identified during the dataset-wide inspection, producing recordings containing **2,048 samples** (approximately 20 seconds).

Using the same participant allows the comparison to focus on the effect of the recording protocol while maintaining consistent subject characteristics.

In [72]:
#7.2.1 Load the recordings
patient_id = (
    short_left_file.name
    .split("_", maxsplit=1)[0]
)

left_long = (
    MOVEMENT_PATH
    / f"{patient_id}_Entrainment_LeftWrist.txt"
)

right_long = (
    MOVEMENT_PATH
    / f"{patient_id}_Entrainment_RightWrist.txt"
)


assert left_long.exists()
assert right_long.exists()


left_long_signal = load_raw_signal(
    left_long
)

right_long_signal = load_raw_signal(
    right_long
)


print("Left wrist :", left_long.name)
print("Right wrist:", right_long.name)

Left wrist : 328_Entrainment_LeftWrist.txt
Right wrist: 328_Entrainment_RightWrist.txt


In [73]:
#7.2.2 Long-duration paired timing summary
long_pair_summary = (
    quality_df[
        quality_df["file"].isin(
            [
                left_long.name,
                right_long.name,
            ]
        )
    ]
    .set_index("wrist")[
        [
            "samples",
            "duration_seconds",
            "sampling_frequency_mean_hz",
            "sampling_frequency_median_hz",
            "jitter_std_ms",
            "minimum_interval_seconds",
            "maximum_interval_seconds",
        ]
    ]
    .T
)


long_pair_summary

wrist,Left,Right
samples,2048.000000,2048.000000
duration_seconds,20.459780,20.606161
sampling_frequency_mean_hz,100.049953,99.339221
sampling_frequency_median_hz,100.054962,99.325188
jitter_std_ms,0.296433,0.256456
minimum_interval_seconds,0.005807,0.006063
maximum_interval_seconds,0.014233,0.014221


In [75]:
#7.2.3 Accelerometer Comparison
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=left_long_signal["time"],
        y=left_long_signal["acc_x"],
        mode="lines",
        name="Left Wrist",
    )
)

fig.add_trace(
    go.Scatter(
        x=right_long_signal["time"],
        y=right_long_signal["acc_x"],
        mode="lines",
        name="Right Wrist",
    )
)

fig.update_layout(
    title=(
        f"Accelerometer X-axis Comparison "
        f"({patient_id} - Entrainment)"
    ),
    xaxis_title="Time (s)",
    yaxis_title="Acceleration",
)

fig.show()

### Observations
Both Entrainment recordings contain 2,048 samples and follow the long-duration acquisition protocol.

The two wrists show similar overall recording lengths, but their durations, interval distributions, and effective sampling frequencies are not assumed to be exactly identical. Consequently, the recordings are considered paired but not yet aligned on a shared sample-level time grid.

Uniform 100 Hz interpolation is required before cross-wrist correlation, phase, lag, or symmetry features are calculated.

In [ ]:
#Gyroscope Comparison
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=left_long_signal["time"],
        y=left_long_signal["gyro_x"],
        mode="lines",
        name="Left Wrist"
    )
)

fig.add_trace(
    go.Scatter(
        x=right_long_signal["time"],
        y=right_long_signal["gyro_x"],
        mode="lines",
        name="Right Wrist"
    )
)

fig.update_layout(
    title=f"Gyroscope X-axis Comparison ({patient_id} - Entrainment)",
    xaxis_title="Time (s)",
    yaxis_title="Angular Velocity"
)

fig.show()

### Observations

The two gyroscope recordings show low-amplitude rotational activity throughout the Entrainment assessment. Their equal sample counts confirm that they follow the same protocol, but the independent timestamp grids prevent a claim of exact sample-level temporal alignment before resampling.

# 8. Conclusions

This notebook examined all 10,318 raw smartwatch recordings in the Parkinson’s Disease Smartwatch Dataset before preprocessing and feature extraction.

The recordings contain the expected seven channels and follow the two protocol-defined lengths of 1,024 and 2,048 samples. Dataset-wide checks also confirmed whether each file matched its metadata-defined sample count and whether missing or infinite sensor values were present.

The expanded timestamp-quality assessment evaluated:

- duplicate timestamps;
- negative and non-increasing intervals;
- intervals substantially below or above the nominal 0.010-second interval;
- mean and median sampling intervals;
- mean- and median-based sampling-frequency estimates;
- timestamp jitter;
- timing differences by task, wrist, and smartwatch model;
- and timing differences between all 5,159 paired left- and right-wrist recordings.

Although recording-level mean sampling frequencies remain close to the nominal 100 Hz protocol, the representative analysis demonstrates that this average can hide individual short and long intervals. In addition, paired left- and right-wrist files should not be described as having identical sample-level time grids solely because they contain the same number of samples.

The raw files and existing CLARABEL-preprocessed files will remain unchanged. Time-domain features may use the current preprocessed samples, while frequency-domain and bilateral features will use signals interpolated onto a common uniform 100 Hz grid.

The complete timestamp-quality, task-level, wrist-level, device-level, paired-timing, and resampling-decision reports were saved as reusable QA outputs.